# Augmenting the BEMO Prompts Dataset

### Loading the BEMO Prompts Dataset & Understanding the Data

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('./datasets/bemo_prompts_dataset_v1.3.csv')

In [3]:
df.sample(5)

,prompt,general,smart_home,todo,mail,learning_resources,others
16,What makes you unique?,0,0,0,0,0,1
335,"What time is it, add a reminder to call the de...",1,0,1,1,0,0
2,Can you understand sarcasm?,0,0,0,0,0,1
342,Send the link to the statistics course and ema...,0,0,0,1,1,0
150,Can you summarize what 'attention' means in ne...,1,0,0,0,1,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 367 entries, 0 to 366
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   prompt              367 non-null    object
 1   general             367 non-null    int64 
 2   smart_home          367 non-null    int64 
 3   todo                367 non-null    int64 
 4   mail                367 non-null    int64 
 5   learning_resources  367 non-null    int64 
 6   others              367 non-null    int64 
dtypes: int64(6), object(1)
memory usage: 20.2+ KB


In [5]:
# Get the sum of numerical columns
general_sum = df['general'].sum()
smart_home_sum = df['smart_home'].sum()
todo_sum = df['todo'].sum()
mail_sum = df['mail'].sum()
learning_resources_sum = df['learning_resources'].sum()
other_sum = df['others'].sum()

print(f"General: {general_sum}")
print(f"Smart Home: {smart_home_sum}")
print(f"Todo: {todo_sum}")
print(f"Mail: {mail_sum}")
print(f"Learning Resources: {learning_resources_sum}")
print(f"Other: {other_sum}")

General: 144
Smart Home: 119
Todo: 149
Mail: 131
Learning Resources: 142
Other: 50


In [6]:
# Shuffle the data
df = df.sample(frac=1).reset_index(drop=True)

In [7]:
prompts = df['prompt'].values.tolist()
prompts[:5]

['What’s new in OpenAI’s API, add a task to integrate it, and email me the changelog?',
 'Can you understand sarcasm?',
 "What's the latest news, set the thermostat to 22 degrees, and compose an email to Ahmed.",
 'How do you spend your time?',
 'Set a reminder to finish the PyTorch course and recommend another one.']

In [8]:
labels = df.drop(columns=["prompt"]).values.tolist()
labels[:5]

[[1, 0, 1, 1, 1, 0],
 [0, 0, 0, 0, 0, 1],
 [1, 1, 0, 1, 0, 0],
 [0, 0, 0, 0, 0, 1],
 [0, 0, 1, 0, 1, 0]]

In [16]:
# Replace each 1 in the labels with the corresponding column name
def replace_labels(row):
    labels = []
    if row['general'] == 1:
        labels.append('general')
    if row['smart_home'] == 1:
        labels.append('smart_home')
    if row['todo'] == 1:
        labels.append('todo')
    if row['mail'] == 1:
        labels.append('mail')
    if row['learning_resources'] == 1:
        labels.append('learning_resources')
    if row['others'] == 1:
        labels.append('others')
    return labels

df['labels'] = df.apply(replace_labels, axis=1)

In [18]:
df.sample(5)

,prompt,general,smart_home,todo,mail,learning_resources,others,labels
206,Activate the security alarm.,0,1,0,0,0,0,[smart_home]
289,Where can I learn about secure IoT setups for ...,0,1,0,0,1,0,"[smart_home, learning_resources]"
201,Teach me how to use Node-RED with my smart swi...,0,1,0,0,1,0,"[smart_home, learning_resources]"
129,Where can I find a good restaurant and send an...,1,0,0,1,0,0,"[general, mail]"
321,"What are the benefits of meditation, set a rem...",1,0,1,1,0,0,"[general, todo, mail]"


In [19]:
# Get the named labels
named_labels = df['labels'].values.tolist()
named_labels[:5]

[['general', 'todo', 'mail', 'learning_resources'],
 ['others'],
 ['general', 'smart_home', 'mail'],
 ['others'],
 ['todo', 'learning_resources']]

### Paraphrasing using gemini-1.5-flash

In [20]:
import os
from dotenv import load_dotenv, find_dotenv, dotenv_values

load_dotenv()
os.environ["HUGGINGFACEHUB_API_TOKEN"] = dotenv_values(find_dotenv())[
    "HUGGINGFACE_API_KEY_TEST" #! Test
]
gemini_api_key = dotenv_values(find_dotenv())["GEMINI_API_KEY_TEST"]  #! Test
os.environ["GOOGLE_API_KEY"] = gemini_api_key 

In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

system_prompt =  ChatPromptTemplate(
            [
                (
                    "system",
                    """
You are an advanced paraphrasing model. Your task is to generate coherent, meaningful paraphrased versions of the given sentence. Ensure the paraphrased sentences retain the original intent, meaning, and context of the input while matching the provided labels.

### Instructions:
1. Paraphrase the sentence into as many variations as you find sufficient to cover diverse phrasings.
2. Keep all paraphrases clear, grammatically correct, and contextually relevant.
3. The paraphrased sentences must still be suitable for the original label(s).

### Format:
Return the output as a JSON object with this structure:
{{"result": ["sentence 1", "sentence 2", "sentence 3", ...]}}

### Example Input:
Sentence: "Remind me to call John and send an email to Ahmed."
Labels: ["todo", "mail"]

### Example Output:
{{
  "result": [
    "Set a reminder for me to call John and email Ahmed.",
    "Don't forget to remind me to call John and shoot an email to Ahmed.",
    "Please remind me about my call with John and to send Ahmed an email.",
    "Help me remember to phone John and draft an email to Ahmed."
  ]
}}

### Now, paraphrase the following:
""",
),
("human", "{user_input}"),
("placeholder", "{messages}"),
            ]
        )

In [23]:
chain = system_prompt | llm | JsonOutputParser()

In [24]:
def gemini_paraphrase(sentence, labels):
    input_dict = {"sentence": sentence, "labels": labels}
    output = chain.invoke(input={"user_input": str(input_dict)})
    return output    

In [25]:
gemini_paraphrase(prompts[0], named_labels[0])

{'result': ["I need to know what's been updated in the OpenAI API; please add a task to integrate those changes and send me the changelog via email.",
  'Update me on the latest OpenAI API features, add a task for integration, and email me the release notes.',
  'Please inform me of any new developments in the OpenAI API.  Create a task to integrate these updates and send the changelog to my email address.',
  'What are the recent changes to the OpenAI API?  Add a task to integrate them and email me the changelog.',
  'Could you provide an update on the OpenAI API, schedule a task for integration, and email me the changes?',
  "Let me know about the OpenAI API's new features.  Create a to-do item to integrate them and send the changelog to my email.",
  'Give me the OpenAI API changelog, add a task for integration, and email it to me.',
  'I need the OpenAI API changelog emailed to me; please also add a task to integrate the new features.']}

In [15]:
# #! RUN THIS ONLY ONCE
last_i = 0
gemini_paraphrased_prompts = []
gemini_paraphrased_labels = []

In [51]:
for i in range(last_i, len(prompts)):
    print(f"Processing prompt {i+1}/{len(prompts)}")
    try:
        results = gemini_paraphrase(prompts[i], named_labels[i])["result"]
    except Exception as e:
        print(f"Error: {e}, Stopping at prompt {i}")
        last_i = i
        break
    gemini_paraphrased_prompts.extend(results)
    for _ in range(len(results)):
        gemini_paraphrased_labels.append(labels[i])

Processing prompt 317/367
Processing prompt 318/367
Processing prompt 319/367
Processing prompt 320/367
Processing prompt 321/367
Processing prompt 322/367
Processing prompt 323/367
Processing prompt 324/367
Processing prompt 325/367
Processing prompt 326/367
Processing prompt 327/367
Processing prompt 328/367
Processing prompt 329/367
Processing prompt 330/367
Processing prompt 331/367
Processing prompt 332/367
Processing prompt 333/367
Processing prompt 334/367
Processing prompt 335/367
Processing prompt 336/367
Processing prompt 337/367
Processing prompt 338/367
Processing prompt 339/367
Processing prompt 340/367
Processing prompt 341/367
Processing prompt 342/367
Processing prompt 343/367
Processing prompt 344/367
Processing prompt 345/367
Processing prompt 346/367
Processing prompt 347/367
Processing prompt 348/367
Processing prompt 349/367
Processing prompt 350/367
Processing prompt 351/367
Processing prompt 352/367
Processing prompt 353/367
Processing prompt 354/367
Processing p

In [52]:
if i == len(prompts) - 1:
    print("All prompts processed successfully")
    last_i = i + 1
else:
    print(f"Processed {i} prompts successfully, {len(prompts) - i} remaining")

All prompts processed successfully


In [53]:
print(len(gemini_paraphrased_prompts), len(gemini_paraphrased_labels))

2689 2689


In [54]:
prompts[:5]

['What’s new in OpenAI’s API, add a task to integrate it, and email me the changelog?',
 'Can you understand sarcasm?',
 "What's the latest news, set the thermostat to 22 degrees, and compose an email to Ahmed.",
 'How do you spend your time?',
 'Set a reminder to finish the PyTorch course and recommend another one.']

In [55]:
labels[:5]

[[1, 0, 1, 1, 1, 0],
 [0, 0, 0, 0, 0, 1],
 [1, 1, 0, 1, 0, 0],
 [0, 0, 0, 0, 0, 1],
 [0, 0, 1, 0, 1, 0]]

In [56]:
gemini_paraphrased_prompts[:5]

["I need to know what's been updated in the OpenAI API; please add a task to integrate those changes and send me the changelog via email.",
 'Update me on the latest OpenAI API features, add a task for integration, and email me the release notes.',
 'Please inform me of any new developments in the OpenAI API.  Create a task to integrate these updates and send the changelog to my email address.',
 "What are the recent changes to the OpenAI API?  Add a task to integrate them and email me the changelog when it's ready.",
 'Could you provide an update on the OpenAI API, schedule a task for integration, and email me the updated documentation?']

In [57]:
gemini_paraphrased_labels[:5]

[[1, 0, 1, 1, 1, 0],
 [1, 0, 1, 1, 1, 0],
 [1, 0, 1, 1, 1, 0],
 [1, 0, 1, 1, 1, 0],
 [1, 0, 1, 1, 1, 0]]

In [58]:
print(last_i)
print(len(prompts) - last_i)

367
0


In [65]:
# Create a new dataframe
df_gemini = pd.DataFrame(
    {
        "prompt": gemini_paraphrased_prompts,
        "general": [label[0] for label in gemini_paraphrased_labels],
        "smart_home": [label[1] for label in gemini_paraphrased_labels],
        "todo": [label[2] for label in gemini_paraphrased_labels],
        "mail": [label[3] for label in gemini_paraphrased_labels],
        "learning_resources": [label[4] for label in gemini_paraphrased_labels],
        "others": [label[5] for label in gemini_paraphrased_labels],
    }
)

In [66]:
df_gemini.sample(5)

,prompt,general,smart_home,todo,mail,learning_resources,others
1044,Describe the history of the Roman civilization.,1,0,0,0,0,0
29,I need a reminder to complete the PyTorch cour...,0,0,1,0,1,0
904,Schedule 'finishing the project' in my tasks a...,0,0,1,1,0,0
232,I need to learn about context handling in tran...,1,0,1,0,1,0
2591,Add learning about smart home security to my t...,1,1,1,0,1,0


In [67]:
df_gemini.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2689 entries, 0 to 2688
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   prompt              2689 non-null   object
 1   general             2689 non-null   int64 
 2   smart_home          2689 non-null   int64 
 3   todo                2689 non-null   int64 
 4   mail                2689 non-null   int64 
 5   learning_resources  2689 non-null   int64 
 6   others              2689 non-null   int64 
dtypes: int64(6), object(1)
memory usage: 147.2+ KB


In [68]:
df_gemini.describe()

,general,smart_home,todo,mail,learning_resources,others
count,2689.000000,2689.000000,2689.000000,2689.000000,2689.000000,2689.000000
mean,0.396058,0.317962,0.404239,0.340647,0.384901,0.150614
std,0.489168,0.465771,0.490836,0.474015,0.486663,0.357738
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [69]:
# Get the sum of numerical columns
general_sum = df_gemini['general'].sum()
smart_home_sum = df_gemini['smart_home'].sum()
todo_sum = df_gemini['todo'].sum()
mail_sum = df_gemini['mail'].sum()
learning_resources_sum = df_gemini['learning_resources'].sum()
other_sum = df_gemini['others'].sum()

print(f"General: {general_sum}")
print(f"Smart Home: {smart_home_sum}")
print(f"Todo: {todo_sum}")
print(f"Mail: {mail_sum}")
print(f"Learning Resources: {learning_resources_sum}")
print(f"Other: {other_sum}")

General: 1065
Smart Home: 855
Todo: 1087
Mail: 916
Learning Resources: 1035
Other: 405


In [70]:
# Load the dataframes from the CSV file
df_og = pd.read_csv("./datasets/bemo_prompts_dataset_v1.3.csv")

In [71]:
# Concatenate the dataframes
df_concat = pd.concat([df_og, df_gemini])

In [72]:
df_concat.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3056 entries, 0 to 2688
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   prompt              3056 non-null   object
 1   general             3056 non-null   int64 
 2   smart_home          3056 non-null   int64 
 3   todo                3056 non-null   int64 
 4   mail                3056 non-null   int64 
 5   learning_resources  3056 non-null   int64 
 6   others              3056 non-null   int64 
dtypes: int64(6), object(1)
memory usage: 191.0+ KB


In [73]:
df_concat.describe()

,general,smart_home,todo,mail,learning_resources,others
count,3056.000000,3056.000000,3056.000000,3056.000000,3056.000000,3056.000000
mean,0.395615,0.318717,0.404450,0.342605,0.385144,0.148887
std,0.489062,0.466055,0.490866,0.474658,0.486709,0.356036
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [74]:
# Save the concatenated dataframe to a CSV file
df_concat.to_csv("./datasets/bemo_prompts_dataset_v2.1.csv")